In [0]:
SELECT state_abbr, COUNT(*)
FROM emission_df2
GROUP BY state_abbr
HAVING COUNT(*) >30
ORDER BY COUNT(*) DESC;

/* 1- State with highest number of emission records */
SELECT 
    state_abbr,
    SUM(ghg_emissions_mtco2e) AS Total_Emission
FROM emission_df2
GROUP BY state_abbr
ORDER BY Total_Emission DESC
LIMIT 10;


/* 2) Which countries have above-average greenhouse Gas emissions? */

WITH avg_emission AS (
    SELECT AVG(ghg_emissions_mtco2e) AS avg_E
    FROM emission_df2
)
SELECT
    county_name,
    state_abbr,
    ghg_emissions_mtco2e
FROM emission_df2, avg_emission
WHERE ghg_emissions_mtco2e > avg_E
ORDER BY ghg_emissions_mtco2e ASC;



/* 2.1) Which States have above-average greenhouse Gas emissions? */

WITH avg_emission AS (
    SELECT AVG(ghg_emissions_mtco2e) AS avg_E
    FROM emission_df2
)
SELECT
    state_abbr AS STATES,
    ghg_emissions_mtco2e AS Total_Emission
FROM emission_df2, avg_emission
WHERE ghg_emissions_mtco2e > avg_E
GROUP BY ghg_emissions_mtco2e, state_abbr
ORDER BY ghg_emissions_mtco2e ASC;


WITH states_avg AS (
    SELECT state_abbr,
    AVG(ghg_emissions_mtco2e) AS avg_states
    FROM emission_df2
    GROUP BY state_abbr
)
SELECT *, 
    (SELECT AVG(ghg_emissions_mtco2e) FROM emission_df2) AS avg_emission  
FROM states_avg
WHERE avg_states > (SELECT AVG(ghg_emissions_mtco2e) 
FROM emission_df2)
ORDER BY avg_states DESC;

/* 2.2) Same concepts... but using conty_state_labl */

WITH COUNTY_AVG AS (
    SELECT county_state_label,
    AVG(ghg_emissions_mtco2e) AS S_C_AVG
    FROM emission_df2
    GROUP BY county_state_label
)
SELECT *,
    (SELECT AVG(ghg_emissions_mtco2e) FROM emission_df2) AS S_C_AVG
FROM county_avg
WHERE S_C_AVG > (SELECT AVG(ghg_emissions_mtco2e) FROM emission_df2)
ORDER BY S_C_AVG DESC;

/* 3) Which countries have above-average greenhouse Gas emissions? */
/*(any county with above average emissions) */
CREATE OR REPLACE TEMP VIEW high_emissions_counties AS (
    SELECT *
    FROM emission_df2
    WHERE ghg_emissions_mtco2e > (SELECT AVG(ghg_emissions_mtco2e) FROM emission_df2)
)



/* 4) What are the highest-emissions counties within each state? */
WITH ranked_counties AS (
    SELECT
    state_abbr,
    county_name,
    ghg_emissions_mtco2e,
    ROW_NUMBER() OVER(
        PARTITION BY state_abbr
        ORDER BY ghg_emissions_mtco2e DESC) AS RN
FROM emission_df2
)
SELECT * FROM ranked_counties
WHERE RN <=3
ORDER BY state_abbr, ghg_emissions_mtco2e DESC;


/* SECOND PART of CREATING and ANSWERING BUSINESS QUESTIONS */

/* Wwhich states have the highest totla emissioions, andn can I that aside so ii  don't have to recompute it every time I need it */
/*QuestION: how many temp tables can I create? */
CREATE OR REPLACE TEMP TABLE tem_state_totals AS (
    SELECT 
    state_abbr,
    county_state_label,
    ghg_emissions_mtco2e,
    population,
    SUM(ghg_emissions_mtco2e) OVER( PARTITION BY county_state_label ORDER BY ghg_emissions_mtco2e DESC) AS EM_Accumulation,
    SUM(population) OVER( PARTITION BY county_state_label ORDER BY population DESC) AS Total_population
FROM emission_df2
);

SELECT * FROM tem_state_totals
ORDER BY EM_Accumulation DESC;



SELECT
    state_id,
    county_id,
    state_abbr,
    county_state_label,
    ghg_emissions_mtco2e,
    SUM(ghg_emissions_mtco2e) OVER(ORDER BY ghg_emissions_mtco2e ASC) AS EM_Accumulation,
    population,
    SUM(population) OVER(ORDER BY population ASC) AS Total_population
FROM emission_df2
ORDER BY ghg_emissions_mtco2e, EM_accumulation;



/* 2) QUESTION2: How does emissions intensity per capita vary across DOE climate zones - are colder/hotter zones
disproportionately high emitters? */
SELECT 
    state_abbr,
    climate_zone,
    COUNT(*) AS num_counties,
    SUM(ghg_emissions_mtco2e) AS TOTAL_EMISSION,
    ROUND(AVG(ghg_emissions_mtco2e), 3) AS avg_emission,
    ROUND(SUM(ghg_emissions_mtco2e)/SUM(population)*1000, 4) AS EM_per_capita
FROM emission_df2
GROUP BY state_abbr, climate_zone
ORDER BY EM_Per_capita;


/* 3) - What percentage of total U.S. emissions does each state actually contribute? */

WITH
    state_emissions AS (
        SELECT state_abbr,
        SUM(ghg_emissions_mtco2e) AS state_total
        FROM emission_df2
        GROUP BY state_abbr
    ),
    us_total AS (
        SELECT SUM(state_total) AS national_total
        FROM state_emissions
    )
SELECT 
    se.state_abbr,
    se.state_total,
    round(se.state_total/ut.national_total*100, 2) AS pct_of_national
FROM state_emissions se
CROSS JOIN us_total ut
ORDER BY pct_of_national DESC;
*/

/* second way */
WITH state_emissions AS (
    SELECT
        state_abbr,
        SUM(ghg_emissions_mtco2e) AS state_total
    FROM emission_df2
    GROUP BY state_abbr
)
SELECT
    se.state_abbr,
    se.state_total,
    ROUND(se.state_total /SUM(e.ghg_emissions_mtco2e)*100, 3) AS pct_of_national
FROM state_emissions se, emission_df2 e
GROUP BY se.state_abbr, se.state_total
ORDER BY pct_of_national DESC; 



/* Within each state, which single county emits the most - and how concentrated is national emissions amoung
the top-emitting counties overall? */
WITH county_ranked AS (
    SELECT 
        state_abbr,
        county_name,
        ghg_emissions_mtco2e,
        SUM(ghg_emissions_mtco2e) OVER (ORDER BY ghg_emissions_mtco2e DESC)
            / SUM(ghg_emissions_mtco2e) OVER() * 100 AS cum_pct_national,
        RANK() OVER (PARTITION BY state_abbr ORDER BY ghg_emissions_mtco2e DESC) AS RN_order
    FROM emission_df2
)

SELECT * FROM county_ranked
WHERE RN_order <=2
ORDER BY county_name DESC;

